# Plain MF vs. Ours (joint uncertainty factorisation)

Implements proposal Sec. 4.1's two models on MovieLens-100K, using the shared
`recsys_loader` / `abstention` / `evaluate` modules from the rest of the repo:

- **Plain MF**: `S = A B^T`, minimising ordinary regularised MSE (Koren et al.).
- **Ours**: `S = A B^T`, `U = C D^T` (eq. 4), trained jointly with

  `L(A,B,C,D) = 1/|Ω| * sum_{(i,j)∈Ω} (S_ij - R_ij)^2 * exp(-U_ij) + λ * sum_{(i,j)∈Ω} U_ij`  (eq. 5)

  Larger `U_ij` = the model is less confident, and down-weights that entry's
  contribution to the rating loss while paying a `λ` penalty for doing so.

Both models get the same hyperparameter search budget (grid size, epochs,
batch size) per proposal Sec. 4.2's fairness requirement, then the winning
configs are refit on train+val and scored on test with `evaluate.sweep(...)`
at the abstention rates from Table 1 (`p = 0, 0.05, 0.10, 0.20`), against the
two naive abstention rules in `abstention.py` (random, low-support).

Set `FAST_SMOKE_TEST = True` below (or the env var `MF_SMOKE_TEST=1`) to
sanity-check the whole pipeline in a few seconds. Leave it `False` to run the
real grid — that's the one meant to actually load your CPU.


In [ ]:
import os

FAST_SMOKE_TEST = os.environ.get("MF_SMOKE_TEST") == "1"


## Setup: threads, paths, imports

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

N_THREADS = os.cpu_count() or 1
torch.set_num_threads(N_THREADS)
torch.set_num_interop_threads(max(1, N_THREADS // 2))
print(f"CPU cores detected: {N_THREADS}")
print(f"torch intra-op threads: {torch.get_num_threads()} | inter-op threads: {torch.get_num_interop_threads()}")

ROOT = Path.cwd().resolve().parent  # theGreatTry/week3 -> theGreatTry
sys.path.append(str(ROOT / "dataset loaders and cleaners"))
sys.path.append(str(ROOT / "experiments"))

from recsys_loader import load_split
from abstention import retain_indices
from evaluate import sweep, ABSTENTION_RATES


## Load MovieLens-100K (shared 70/20/10 protocol, seed=42)

In [ ]:
SEED = 42
DATASET = "ml-100k"

split = load_split(DATASET, seed=SEED)
print(f"{split.name}: {split.n_users:,} users x {split.n_items:,} items")
print(f"train={len(split.train):,}  val={len(split.val):,}  test={len(split.test):,}")

device = torch.device("cpu")


def to_tensors(rows, device):
    u = torch.as_tensor(rows[:, 0], dtype=torch.long, device=device)
    i = torch.as_tensor(rows[:, 1], dtype=torch.long, device=device)
    r = torch.as_tensor(rows[:, 2], dtype=torch.float32, device=device)
    return u, i, r


train_u, train_i, train_r = to_tensors(split.train, device)
val_u, val_i, val_r = to_tensors(split.val, device)
test_u, test_i, test_r = to_tensors(split.test, device)

trainval_rows = np.concatenate([split.train, split.val], axis=0)
trainval_u, trainval_i, trainval_r = to_tensors(trainval_rows, device)


## Shared minibatch helper + RMSE

In [ ]:
def minibatches(n, batch_size, generator):
    perm = torch.randperm(n, generator=generator)
    for start in range(0, n, batch_size):
        yield perm[start : start + batch_size]


def rmse(pred, actual):
    return torch.sqrt(torch.mean((pred - actual) ** 2)).item()


## Model 1: Plain MF

In [ ]:
class PlainMF(nn.Module):
    def __init__(self, n_users, n_items, k, seed):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.A = nn.Parameter(torch.randn(n_users, k, generator=g) * 0.1)
        self.B = nn.Parameter(torch.randn(n_items, k, generator=g) * 0.1)

    def forward(self, u, i):
        return (self.A[u] * self.B[i]).sum(dim=1)


def train_plain_mf(
    tr_u, tr_i, tr_r, va_u, va_i, va_r,
    n_users, n_items, k, lr, weight_decay, epochs, batch_size, seed,
    desc=None, show_progress=True,
):
    model = PlainMF(n_users, n_items, k, seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    gen = torch.Generator().manual_seed(seed)

    history = []
    bar = tqdm(range(epochs), desc=desc or f"PlainMF k={k} lr={lr} wd={weight_decay}",
               leave=False, disable=not show_progress)
    for _ in bar:
        for idx in minibatches(len(tr_r), batch_size, gen):
            opt.zero_grad()
            pred = model(tr_u[idx], tr_i[idx])
            loss = torch.mean((pred - tr_r[idx]) ** 2)
            loss.backward()
            opt.step()

        with torch.no_grad():
            v_rmse = rmse(model(va_u, va_i), va_r)
        history.append(v_rmse)
        if show_progress:
            bar.set_postfix(val_rmse=f"{v_rmse:.4f}")

    return model, history


## Model 2: Ours (joint rating + uncertainty factorisation)

The regulariser `λ * sum_Ω U_ij` in eq. (5) is a sum over the *whole* training
set, not a mean — so when we only see one minibatch at a time, we scale that
term by `|Ω_train| / batch_size` to keep it an unbiased estimate of the full
sum (otherwise `λ` would need re-tuning every time the batch size changes).


In [ ]:
class OursModel(nn.Module):
    """S = A B^T (rating head), U = C D^T (uncertainty head) - proposal eq. (4)."""

    def __init__(self, n_users, n_items, k, seed):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.A = nn.Parameter(torch.randn(n_users, k, generator=g) * 0.1)
        self.B = nn.Parameter(torch.randn(n_items, k, generator=g) * 0.1)
        self.C = nn.Parameter(torch.randn(n_users, k, generator=g) * 0.1)
        self.D = nn.Parameter(torch.randn(n_items, k, generator=g) * 0.1)

    def forward(self, u, i):
        s = (self.A[u] * self.B[i]).sum(dim=1)
        u_hat = (self.C[u] * self.D[i]).sum(dim=1)
        return s, u_hat


def joint_loss(s, u_hat, actual, lam, full_n):
    """Proposal eq. (5): mean((S-R)^2 * exp(-U)) + lam * sum_Omega(U), the second
    term estimated from this minibatch and rescaled to the full training set."""
    mse_term = torch.mean((s - actual) ** 2 * torch.exp(-u_hat))
    reg_term = lam * u_hat.sum() * (full_n / len(u_hat))
    return mse_term + reg_term


def train_ours(
    tr_u, tr_i, tr_r, va_u, va_i, va_r,
    n_users, n_items, k, lr, lam, epochs, batch_size, seed,
    desc=None, show_progress=True,
):
    model = OursModel(n_users, n_items, k, seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    gen = torch.Generator().manual_seed(seed)
    full_n = len(tr_r)

    history = []
    bar = tqdm(range(epochs), desc=desc or f"Ours k={k} lr={lr} lam={lam:.0e}",
               leave=False, disable=not show_progress)
    for _ in bar:
        for idx in minibatches(full_n, batch_size, gen):
            opt.zero_grad()
            s, u_hat = model(tr_u[idx], tr_i[idx])
            loss = joint_loss(s, u_hat, tr_r[idx], lam, full_n)
            loss.backward()
            opt.step()

        with torch.no_grad():
            s_val, _ = model(va_u, va_i)
            v_rmse = rmse(s_val, va_r)
        history.append(v_rmse)
        if show_progress:
            bar.set_postfix(val_rmse=f"{v_rmse:.4f}")

    return model, history


## Hyperparameter grid (matched search budget between the two methods)

Both methods get the same `(k, lr)` grid and the same number of regularisation
values (`weight_decay` for Plain MF, `λ` for Ours), so neither gets an unfair
head start per proposal Sec. 4.2. This is also the CPU-burning part: every
`(config, epoch, minibatch)` triple is a real forward+backward pass.


In [ ]:
if FAST_SMOKE_TEST:
    K_GRID = [10]
    LR_GRID = [0.01]
    WD_GRID = [1e-2]
    LAM_GRID = [1e-4]
    EPOCHS = 3
    BATCH_SIZE = 4096
else:
    K_GRID = [10, 20, 50, 100]
    LR_GRID = [0.003, 0.01]
    WD_GRID = [1e-3, 5e-3, 1e-2, 5e-2]
    LAM_GRID = [1e-6, 1e-5, 1e-4, 1e-3]
    EPOCHS = 400
    BATCH_SIZE = 256

mf_configs = [{"k": k, "lr": lr, "weight_decay": wd} for k in K_GRID for lr in LR_GRID for wd in WD_GRID]
ours_configs = [{"k": k, "lr": lr, "lam": lam} for k in K_GRID for lr in LR_GRID for lam in LAM_GRID]

total_runs = len(mf_configs) + len(ours_configs)
total_steps = total_runs * EPOCHS * -(-len(train_r) // BATCH_SIZE)
print(f"Plain MF configs: {len(mf_configs)} | Ours configs: {len(ours_configs)} | epochs each: {EPOCHS}")
print(f"~{total_steps:,} total optimiser steps across the whole grid search")


## Grid search: Plain MF

In [ ]:
mf_results = []
pbar = tqdm(mf_configs, desc="Plain MF grid search")
t0 = time.time()
for cfg in pbar:
    model, history = train_plain_mf(
        train_u, train_i, train_r, val_u, val_i, val_r,
        split.n_users, split.n_items, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
        show_progress=False, **cfg,
    )
    mf_results.append({**cfg, "val_rmse": history[-1], "model": model, "history": history})
    pbar.set_postfix(val_rmse=f"{history[-1]:.4f}")
print(f"Plain MF grid search took {time.time() - t0:.1f}s")

best_mf = min(mf_results, key=lambda r: r["val_rmse"])
print("Best Plain MF config:", {k: v for k, v in best_mf.items() if k not in ("model", "history")})


## Grid search: Ours

In [ ]:
ours_results = []
pbar = tqdm(ours_configs, desc="Ours grid search")
t0 = time.time()
for cfg in pbar:
    model, history = train_ours(
        train_u, train_i, train_r, val_u, val_i, val_r,
        split.n_users, split.n_items, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED,
        show_progress=False, **cfg,
    )
    ours_results.append({**cfg, "val_rmse": history[-1], "model": model, "history": history})
    pbar.set_postfix(val_rmse=f"{history[-1]:.4f}")
print(f"Ours grid search took {time.time() - t0:.1f}s")

best_ours = min(ours_results, key=lambda r: r["val_rmse"])
print("Best Ours config:", {k: v for k, v in best_ours.items() if k not in ("model", "history")})


## Refit winning configs on train+val, then touch test once

In [ ]:
final_mf, _ = train_plain_mf(
    trainval_u, trainval_i, trainval_r, test_u, test_i, test_r,
    split.n_users, split.n_items,
    k=best_mf["k"], lr=best_mf["lr"], weight_decay=best_mf["weight_decay"],
    epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, desc="Refit Plain MF",
)

final_ours, _ = train_ours(
    trainval_u, trainval_i, trainval_r, test_u, test_i, test_r,
    split.n_users, split.n_items,
    k=best_ours["k"], lr=best_ours["lr"], lam=best_ours["lam"],
    epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, desc="Refit Ours",
)


## Selective RMSE (Table 1 rows we can fill in from this notebook)

`Ours (Û)` uses the model's own learned uncertainty as the unreliability
score; the two naive rules (`random`, `low-support`, from `abstention.py`)
score Plain MF's predictions, since Plain MF has no built-in uncertainty
signal. `IGMC` / `SoftImpute` / `AutoRec` / `UAIMC` rows live in `week2/` and
aren't wired into this shared table yet.


In [ ]:
actual = split.test[:, 2]
test_user_idx = split.test[:, 0].astype(int)
train_user_idx_ = split.train[:, 0].astype(int)

with torch.no_grad():
    mf_pred = final_mf(test_u, test_i).numpy()
    ours_pred, ours_uhat = final_ours(test_u, test_i)
    ours_pred = ours_pred.numpy()
    ours_uhat = ours_uhat.numpy()

rng = np.random.default_rng(SEED)
random_score = rng.random(len(actual))

n_users_total = int(max(train_user_idx_.max(), test_user_idx.max())) + 1
train_support = np.bincount(train_user_idx_, minlength=n_users_total)
support_score = -train_support[test_user_idx].astype(np.float64)

table = {
    "Plain MF (rand)": sweep(mf_pred, actual, random_score),
    "Plain MF (supp)": sweep(mf_pred, actual, support_score),
    "Ours (U_hat)": sweep(ours_pred, actual, ours_uhat),
}

rows = []
for name, r in table.items():
    row = {"method": name}
    row.update({f"p={p}": round(v, 4) for p, v in r.items()})
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("method")
results_df


## Plot: selective RMSE vs. abstention rate

In [ ]:
plt.figure(figsize=(7, 5))
for name, r in table.items():
    xs = sorted(r.keys())
    ys = [r[x] for x in xs]
    plt.plot(xs, ys, marker="o", label=name)
plt.xlabel("Abstention rate p")
plt.ylabel("Selective RMSE")
plt.title(f"Selective RMSE vs abstention rate ({DATASET})")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Notes / next steps

- **More CPU to burn**: bump `EPOCHS`, widen `K_GRID`/`WD_GRID`/`LAM_GRID`, or
  switch `DATASET` to `"ml-1m"` / `"ml-25m"` — everything above is written
  against `recsys_loader.load_split`, so that's a one-line change.
- Proposal Sec. 4.2 also wants a **matched-`k`, matched-`µS`, matched-budget**
  zero-abstention comparison specifically isolating whether the uncertainty
  term itself improves the rating head (not just picking whatever config wins
  validation RMSE independently per method) — that's a follow-up, not done
  here yet.
- IGMC / SoftImpute / AutoRec / UAIMC baselines are in `week2/`; hooking their
  predictions + an unreliability score into the same `table`/`results_df`
  pattern here would fill in the rest of proposal Table 1.
